In [54]:
from databricks import sql
import pandas as pd

conn = sql.connect(
    server_hostname="dbc-fd7ded33-7c60.cloud.databricks.com",
    http_path="/sql/1.0/warehouses/09b2340fc9bd376b"
)

query = """
select 
    distinct(replace(dprod.dc_produto,'+',',')) as marca,
    replace(dprod.dc_molecula_assoc,' + ',',') as alias
from
    auditoria.cdd_m.dm_produtos_corp as dprod
where
    dprod.dc_fabricante ilike 'EMS%' 
    and dprod.dc_corporacao ilike '%NC%'
group by all
"""

cursor = conn.cursor()
cursor.execute(query)

result = cursor.fetchall()

df = pd.DataFrame(
    result,
    columns=[desc[0] for desc in cursor.description]
)

cursor.close()
conn.close()

In [55]:
import unicodedata
from typing import List, Dict, Any
from rapidfuzz import process, fuzz
from collections import defaultdict

# ============================================
# 2. NORMALIZAÇÃO
# ============================================

def normalize(text: str) -> str:
    """
    Normaliza texto:
    - lowercase
    - remove acentos
    - remove espaços extras
    """

    if not isinstance(text, str):
        return ""

    text = text.lower().strip()

    text = unicodedata.normalize("NFKD", text)

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    return " ".join(text.split())



In [63]:

# ============================================
# 3. SCORER PRINCIPAL
# ============================================

def hybrid_score(a, b, **kwargs):

    token_score = fuzz.token_set_ratio(a, b)

    size_ratio = len(a) / max(len(b), 1)

    partial_score = (
        fuzz.partial_ratio(a, b)
        if size_ratio >= 0.6
        else 0
    )

    score = max(token_score, partial_score)

    # penalização para nomes curtos
    if len(b) <= 4:
        score *= 0.75

    return score


In [64]:

# ============================================
# 4. PREPARAÇÃO DOS DADOS
# ============================================

# --------------------------------------------
# 4.1 Produtos únicos
# --------------------------------------------

produtos = (
    df["marca"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

# --------------------------------------------
# 4.2 Aliases explodidos corretamente
# --------------------------------------------

aliases = defaultdict(list)

for marca, alias_str in zip(df["marca"], df["alias"]):

    if pd.isna(alias_str):
        continue

    alias_list = [
        a.strip()
        for a in alias_str.split(",")
        if a.strip()
    ]

    aliases[marca].extend(alias_list)



In [65]:

# ============================================
# 5. PRODUCT SEARCH
# ============================================

class ProductSearch:

    def __init__(
        self,
        produtos: List[str],
        aliases: Dict[str, List[str]]
    ):

        # ------------------------------------
        # Produtos
        # ------------------------------------

        self.produtos = produtos

        self.produtos_norm = [
            normalize(p)
            for p in produtos
        ]

        # índice:
        # produto normalizado -> original

        self.index_map = {
            self.produtos_norm[i]: produtos[i]
            for i in range(len(produtos))
        }

        # ------------------------------------
        # Alias -> Produto
        # ------------------------------------

        self.alias_to_product = {}

        for produto, alias_list in aliases.items():

            produto_norm = normalize(produto)

            for alias in alias_list:

                alias_norm = normalize(alias)

                # evita alias vazio
                if alias_norm:

                    self.alias_to_product[alias_norm] = produto


    # ========================================
    # BUSCA PRINCIPAL
    # ========================================

    def search(
        self,
        query: str,
        top_k: int = 3
    ) -> Dict[str, Any]:

        query_norm = normalize(query)

        # ====================================
        # ETAPA 1 — BUSCA SOMENTE EM MARCAS
        # ====================================

        resultados = process.extract(
            query_norm,
            self.produtos_norm,
            scorer=hybrid_score,
            limit=top_k
        )

        ranking = []

        for match, score, idx in resultados:

            ranking.append({
                "produto": self.produtos[idx],
                "score": float(score),
                "fonte": "marca"
            })

        ranking = sorted(
            ranking,
            key=lambda x: x["score"],
            reverse=True
        )

        # ====================================
        # ETAPA 2 — HIGH CONFIDENCE
        # ====================================

        if ranking and ranking[0]["score"] >= 85:

            top = ranking[0]

            return {
                "status": "high_confidence",
                "mensagem": (
                    f"Encontrei: {top['produto']}. "
                    f"Confirma?"
                ),
                "sugestoes": [top]
            }

        # ====================================
        # ETAPA 3 — FALLBACK POR ALIAS
        # ====================================

        alias_match = process.extractOne(
            query_norm,
            list(self.alias_to_product.keys()),
            scorer=fuzz.token_set_ratio
        )

        if alias_match:

            alias_encontrado, alias_score, _ = alias_match

            # threshold conservador
            if alias_score >= 90:

                produto_sugerido = self.alias_to_product[
                    alias_encontrado
                ]

                return {
                    "status": "alias_suggestion",
                    "mensagem": (
                        f"Você quis dizer "
                        f"{produto_sugerido}?"
                    ),
                    "sugestoes": [
                        {
                            "produto": produto_sugerido,
                            "score": float(alias_score),
                            "fonte": "alias"
                        }
                    ]
                }

        # ====================================
        # ETAPA 4 — NÃO ENCONTREI
        # ====================================

        return {
            "status": "not_found",
            "mensagem": (
                "Não consegui encontrar o medicamento 😕\n"
                "Digite o nome exatamente "
                "como aparece na caixa."
            ),
            "sugestoes": []
        }



In [66]:

# ============================================
# 6. INSTÂNCIA
# ============================================

buscador = ProductSearch(
    produtos=produtos,
    aliases=aliases
)



In [67]:

# ============================================
# 7. TESTES
# ============================================

queries = [

    # Marca correta
    "fluoxetina ems",
    "ENDCOFF",
    "Gelmax",

    # Typos
    "fluoxetinna",
    "endcof",
    "gelmaxe",
    "multigripe",

    # Linguagem natural
    "quero informação sobre fluoxetina",
    "tem bisuran?",
    "preciso do multigrip",

    # Alias
    "amonio cloreto",
    "bromexina",
    "paracetamol",

    # Difíceis
    "aluminio hidroxido",
    "sodio citrato"
]


for q in queries:

    print("\n" + "=" * 60)

    print(f"Query: {q}")

    resultado = buscador.search(q)

    print("Status:", resultado["status"])
    print("Mensagem:", resultado["mensagem"])

    for i, s in enumerate(resultado["sugestoes"], 1):

        print(
            f"{i}. "
            f"{s['produto']} | "
            f"score: {s['score']:.1f} | "
            f"fonte: {s['fonte']}"
        )


Query: fluoxetina ems
Status: high_confidence
Mensagem: Encontrei: FLUOXETINA EMS. Confirma?
1. FLUOXETINA EMS | score: 100.0 | fonte: marca

Query: ENDCOFF
Status: high_confidence
Mensagem: Encontrei: ENDCOFF. Confirma?
1. ENDCOFF | score: 100.0 | fonte: marca

Query: Gelmax
Status: high_confidence
Mensagem: Encontrei: GELMAX. Confirma?
1. GELMAX | score: 100.0 | fonte: marca

Query: fluoxetinna
Status: high_confidence
Mensagem: Encontrei: FLUOXETINA EMS. Confirma?
1. FLUOXETINA EMS | score: 95.2 | fonte: marca

Query: endcof
Status: high_confidence
Mensagem: Encontrei: ENDCOFF. Confirma?
1. ENDCOFF | score: 100.0 | fonte: marca

Query: gelmaxe
Status: high_confidence
Mensagem: Encontrei: GELMAX. Confirma?
1. GELMAX | score: 100.0 | fonte: marca

Query: multigripe
Status: high_confidence
Mensagem: Encontrei: MULTIGRIP. Confirma?
1. MULTIGRIP | score: 100.0 | fonte: marca

Query: quero informação sobre fluoxetina
Status: alias_suggestion
Mensagem: Você quis dizer DAFORIN?
1. DAFORIN |

In [68]:
queries = [

    # ========================================
    # MATCH EXATO
    # ========================================

    "FLUOXETINA EMS",
    "ENDCOFF",
    "BISURAN",
    "GELMAX",
    "MULTIGRIP",

    # ========================================
    # TYPOS LEVES
    # ========================================

    "fluoxetinna",
    "endcof",
    "gelmaxe",
    "multigripe",
    "bisurra",

    # ========================================
    # TYPOS PESADOS
    # ========================================

    "fluxetina",
    "giumax",
    "endicoff",
    "multigripi",
    "bromexsina",

    # ========================================
    # ERROS FONÉTICOS
    # ========================================

    "fluoxcetina",
    "paracetamou",
    "difenidraminna",
    "clorfeniraminaa",

    # ========================================
    # LINGUAGEM NATURAL
    # ========================================

    "quero informação sobre fluoxetina ems",
    "tem bisuran?",
    "preciso do multigrip",
    "me fala sobre gelmax",
    "quero saber se endcoff da sono",

    # ========================================
    # ALIAS DIRETO
    # ========================================

    "amonio cloreto",
    "difenidramina",
    "sodio citrato",
    "bromexina",
    "paracetamol",
    "clorfeniramina",
    "fenilefrina",
    "aluminio hidroxido",
    "calcio carbonato",

    # ========================================
    # ALIAS + FRASE
    # ========================================

    "quero saber sobre bromexina",
    "me explica paracetamol",
    "informação sobre difenidramina",
    "serve para que aluminio hidroxido",

    # ========================================
    # CASE INSENSITIVE
    # ========================================

    "gElMaX",
    "eNdCoFf",
    "fluOXEtina ems",

    # ========================================
    # ACENTUAÇÃO
    # ========================================

    "fluoxetína",
    "paracetamól",
    "clorfeníramina",

    # ========================================
    # ESPAÇOS E RUÍDOS
    # ========================================

    "   gelmax   ",
    ".....endcoff.....",
    "###multigrip###",

    # ========================================
    # NÃO ENCONTRADO
    # ========================================

    "remedio azul",
    "dor de cabeça",
    "gripe forte",
    "antibiotico",
    "abcxyz123",

    # ========================================
    # TESTES PERIGOSOS
    # ========================================

    "cloreto",
    "hidroxido",
    "sodio",
    "calcio",
    "magnesio",

    # ========================================
    # AMBIGUIDADE FARMACÊUTICA
    # ========================================

    "fluoxetina",
    "dipirona",
    "ibuprofeno",
    "paracetamol",

    # ========================================
    # INPUTS REAIS DE USUÁRIO
    # ========================================

    "o remedio gelmax",
    "caixa de multigripe",
    "fluoxetina da ems",
    "xarope endcoff",
    "remedio bisuran",

    # ========================================
    # LIXO / EDGE CASES
    # ========================================

    "",
    " ",
    "123456",
    "@@@@@",
    "kkkkkkkk",
]

for q in queries:

    print("\n" + "=" * 60)

    print(f"Query: {q}")

    resultado = buscador.search(q)

    print("Status:", resultado["status"])
    print("Mensagem:", resultado["mensagem"])

    for i, s in enumerate(resultado["sugestoes"], 1):

        print(
            f"{i}. "
            f"{s['produto']} | "
            f"score: {s['score']:.1f} | "
            f"fonte: {s['fonte']}"
        )


Query: FLUOXETINA EMS
Status: high_confidence
Mensagem: Encontrei: FLUOXETINA EMS. Confirma?
1. FLUOXETINA EMS | score: 100.0 | fonte: marca

Query: ENDCOFF
Status: high_confidence
Mensagem: Encontrei: ENDCOFF. Confirma?
1. ENDCOFF | score: 100.0 | fonte: marca

Query: BISURAN
Status: high_confidence
Mensagem: Encontrei: BISURAN. Confirma?
1. BISURAN | score: 100.0 | fonte: marca

Query: GELMAX
Status: high_confidence
Mensagem: Encontrei: GELMAX. Confirma?
1. GELMAX | score: 100.0 | fonte: marca

Query: MULTIGRIP
Status: high_confidence
Mensagem: Encontrei: MULTIGRIP. Confirma?
1. MULTIGRIP | score: 100.0 | fonte: marca

Query: fluoxetinna
Status: high_confidence
Mensagem: Encontrei: FLUOXETINA EMS. Confirma?
1. FLUOXETINA EMS | score: 95.2 | fonte: marca

Query: endcof
Status: high_confidence
Mensagem: Encontrei: ENDCOFF. Confirma?
1. ENDCOFF | score: 100.0 | fonte: marca

Query: gelmaxe
Status: high_confidence
Mensagem: Encontrei: GELMAX. Confirma?
1. GELMAX | score: 100.0 | fonte: 